In [1]:
import glob
import os
from typing import List
import numpy as np

import imagehash
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image


class CosineSimilarityReward(nn.Module):
    def __init__(
        self,
        model_name="vit_base_patch16_clip_384.laion2b_ft_in12k_in1k",
        threshold=0.6,
    ):
        super(CosineSimilarityReward, self).__init__()
        self.threshold = threshold
        self.model, self.transforms = self.get_model(model_name)

    def get_model(self, model_name):
        model = timm.create_model(model_name, pretrained=True, num_classes=0)
        model.eval()
        data_config = timm.data.resolve_model_data_config(model)
        transforms = timm.data.create_transform(**data_config, is_training=False)
        return model, transforms

    @torch.inference_mode()
    def forward(
        self, validator_image: Image.Image, miner_image: Image.Image
    ) -> tuple[float]:
        validator_vec = self.model(self.transforms(validator_image).unsqueeze(0))
        image_vec = self.model(self.transforms(miner_image).unsqueeze(0))
        cosine_similarity = F.cosine_similarity(validator_vec, image_vec)

        if cosine_similarity.item() > self.threshold:
            reward = 1.0
        elif cosine_similarity.item() > 0.4:
            reward = cosine_similarity.item()
        else:
            reward = 0.0

        return float(cosine_similarity.item()), reward

    def get_reward(
        self, validator_image: Image.Image, miner_image: Image.Image
    ) -> tuple[float]:
        nsfw_check = self.nsfw_filter(validator_image, miner_image)
        if nsfw_check:
            reward_cut = -5 if nsfw_check == 2 else 0
            reward_full = -1
        else:
            reward_full, reward_cut = self.matching_image(miner_image, validator_image)
        return reward_full, reward_cut

    def get_reward_for_folder(self, folder_path: str) -> List[List[float]]:
        original_images_paths = glob.glob(os.path.join(folder_path, "*-original.*"))
        test_images_paths = glob.glob(os.path.join(folder_path, "*-test.*"))
        numbs = [
            int(os.path.basename(file).split("-")[0])
            for file in original_images_paths + test_images_paths
        ]
        start, end = min(numbs), max(numbs) + 1

        rewards = []
        for i in range(start, end):
            try:
                oiginal_image_path = [
                    file for file in original_images_paths if i == int(os.path.basename(file).split("-")[0])
                ][0]
                test_image_path = [
                    file for file in test_images_paths if i == int(os.path.basename(file).split("-")[0])
                ][0]
                original_image = Image.open(oiginal_image_path)
                test_image = Image.open(test_image_path)
                reward_full, reward_cut = self.get_reward(original_image, test_image)
                rewards.append([i, reward_full, reward_cut])
            except IndexError as e:
                print(f"Incomplete pair with number {i}, skipping...")
        return rewards

    def get_black_hash(self, H, W) -> str:
        image = Image.new("RGB", (W, H), color="black")
        return str(imagehash.average_hash(image, hash_size=8))

    def matching_image(
        self, miner_image: Image.Image, validator_image: Image.Image
    ) -> bool:
        cosine_similarity_score_full, cosine_similarity_score_cut = self.forward(
            validator_image, miner_image
        )
        return cosine_similarity_score_full, cosine_similarity_score_cut

    def nsfw_filter(
        self, validator_image: Image.Image, miner_image: Image.Image
    ) -> bool:
        W, H = validator_image.size
        validator_hash = imagehash.average_hash(validator_image, hash_size=6)
        miner_hash = imagehash.average_hash(miner_image, hash_size=6)
        validator_hash = str(validator_hash)
        miner_hash = str(miner_hash)
        black_hash = self.get_black_hash(H, W)
        if validator_hash != black_hash and miner_hash == black_hash:
            return 1
        if validator_hash == black_hash and miner_hash != black_hash:
            return 2
        return 0

    def get_final_reward(
        self, matching_result: float, time_spent: float, total_volume: int
    ) -> float:
        timeout = 12
        time_penalty = 0.4 * (time_spent / timeout) ** 3
        volume_scale = max(min(np.sqrt(total_volume / 1000), 1), 0)

        return (matching_result - time_penalty) * (0.6 + 0.4 * volume_scale)
    

/root/bt-23-inferring/.conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
reward = CosineSimilarityReward()
result = reward.get_reward_for_folder("tests")

In [5]:
# Filter out all result items less then 0.6
new_result = [item for item in result if item[1] > 0.6]

len(new_result), min([item[1] for item in result]), max([item[1] for item in result])

(49, 0.5720430016517639, 0.978427529335022)

In [7]:
[item for item in result if item[1] < 0.6]

[[12, 0.5720430016517639, 0.5720430016517639]]